# 4.1 Task A — Price Prediction (Regressione)\n**Autore:** Studente 1\n**Input:** `outputs/listings_clean.parquet`\n**Output:** `models/price_model.pkl`, `metrics/price_metrics.json`

In [ ]:
import pandas as pd\nimport numpy as np\nimport json\nimport joblib\nimport os\n\nfrom sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, KFold\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.linear_model import Ridge, Lasso\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\n\ntry:\n    from xgboost import XGBRegressor\nexcept ImportError:\n    XGBRegressor = None

## 4.1.1 Caricamento Dati Condivisi

In [ ]:
df = pd.read_parquet("outputs/listings_clean.parquet")\nprint("Shape:", df.shape)

## 4.1.2 Definizione Target e Feature

In [ ]:
TARGET = "price"\n\n# Drop righe con target nullo\ndf = df.dropna(subset=[TARGET])\n\n# LEAKAGE GUARD: escludere occupancy e revenue stimati\nleakage_cols = ["estimated_occupancy_l365d", "estimated_revenue_l365d"]\ndf = df.drop(columns=[c for c in leakage_cols if c in df.columns])

## 4.1.3 Bridge Opzionale: Sentiment Score da Task C

In [ ]:
if os.path.exists("outputs/comment_sentiments.csv"):\n    sentiment = pd.read_csv("outputs/comment_sentiments.csv")\n    sentiment_agg = sentiment.groupby("listing_id")["sentiment_score"].mean().reset_index()\n    df = df.merge(sentiment_agg, left_on="id", right_on="listing_id", how="left")\n    df["sentiment_score"] = df["sentiment_score"].fillna(0.5)\n    print("[BRIDGE] Sentiment score integrato.")\nelse:\n    print("[BRIDGE] Sentiment score non disponibile, procedo senza.")

## 4.1.4 Preparazione X, y e Split

In [ ]:
# TODO: definire numeric_features e categorical_features in base al dataset pulito\nnumeric_features = []   # es. ['accommodates', 'bedrooms', 'beds', ...]\ncategorical_features = []  # es. ['room_type', 'neighbourhood_cleansed', ...]\n\ny = df[TARGET]\nX = df[numeric_features + categorical_features]\n\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y, test_size=0.2, random_state=42\n)

## 4.1.5 Pipeline Preprocessing

In [ ]:
preprocessor = ColumnTransformer(\n    transformers=[\n        ("num", StandardScaler(), numeric_features),\n        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)\n    ]\n)

## 4.1.6 Helper Metriche

In [ ]:
def regression_metrics(name, model, X_test, y_test):\n    y_pred = model.predict(X_test)\n    mae = mean_absolute_error(y_test, y_pred)\n    rmse = mean_squared_error(y_test, y_pred, squared=False)\n    r2 = r2_score(y_test, y_pred)\n    print(f"{name}: MAE={mae:.2f}, RMSE={rmse:.2f}, R2={r2:.4f}")\n    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

## 4.1.7 Modello 1 — Ridge Regression (baseline)

In [ ]:
ridge = Pipeline([\n    ("preprocessor", preprocessor),\n    ("model", Ridge())\n])\nridge.fit(X_train, y_train)\nmetrics_ridge = regression_metrics("Ridge", ridge, X_test, y_test)

## 4.1.8 Modello 2 — LASSO

In [ ]:
lasso = Pipeline([\n    ("preprocessor", preprocessor),\n    ("model", Lasso(max_iter=10000))\n])\nlasso.fit(X_train, y_train)\nmetrics_lasso = regression_metrics("LASSO", lasso, X_test, y_test)

## 4.1.9 Modello 3 — Random Forest

In [ ]:
rf = Pipeline([\n    ("preprocessor", preprocessor),\n    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))\n])\nrf.fit(X_train, y_train)\nmetrics_rf = regression_metrics("RandomForest", rf, X_test, y_test)

## 4.1.10 Modello 4 — XGBoost (scelto per ottimizzazione)

In [ ]:
if XGBRegressor:\n    xgb = Pipeline([\n        ("preprocessor", preprocessor),\n        ("model", XGBRegressor(random_state=42, n_jobs=-1))\n    ])\n    xgb.fit(X_train, y_train)\n    metrics_xgb = regression_metrics("XGBoost", xgb, X_test, y_test)\nelse:\n    print("XGBoost non installato, salto.")\n    metrics_xgb = {"model": "XGBoost", "MAE": None, "RMSE": None, "R2": None}

## 4.1.11 Confronto

In [ ]:
results = [metrics_ridge, metrics_lasso, metrics_rf, metrics_xgb]\npd.DataFrame(results)

## 5.1 Ottimizzazione XGBoost

In [ ]:
if XGBRegressor:\n    param_grid = {\n        "model__n_estimators": [100, 300, 500],\n        "model__max_depth": [3, 5, 7],\n        "model__learning_rate": [0.01, 0.1, 0.3],\n        "model__subsample": [0.8, 1.0],\n    }\n\n    search = RandomizedSearchCV(\n        xgb, param_distributions=param_grid,\n        n_iter=10, cv=5, scoring="neg_root_mean_squared_error",\n        random_state=42, n_jobs=-1\n    )\n    search.fit(X_train, y_train)\n\n    print("Best params:", search.best_params_)\n    metrics_xgb_opt = regression_metrics("XGBoost_Opt", search.best_estimator_, X_test, y_test)\n    best_model = search.best_estimator_\nelse:\n    best_model = ridge  # fallback\n    metrics_xgb_opt = metrics_ridge

## Export

In [ ]:
joblib.dump(best_model, "models/price_model.pkl")\n\nwith open("metrics/price_metrics.json", "w") as f:\n    json.dump({\n        "results": results,\n        "best_model": metrics_xgb_opt if 'metrics_xgb_opt' in dir() else metrics_ridge\n    }, f, indent=2)\n\nprint("Task A completato e esportato.")